# 03. Feature Engineering: Rácios Financeiros e Governance
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Este notebook prepara e valida os indicadores preditivos (Rácios) a partir dos dados limpos.

**Nota Metodológica (Pipeline):** 
Input: `data/interim/micro_long.csv`  
Output: `data/interim/micro_features.csv`

In [11]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')
from src.data.preprocessing import safe_float

DATA_INTERIM = "../data/interim"
df = pd.read_csv(os.path.join(DATA_INTERIM, "micro_long.csv"), low_memory=False)

print(f'Dataset carregado: {len(df):,} observações | {df["NIF Code"].nunique():,} empresas')
print(f'Colunas disponíveis: {list(df.columns)}')
print(f'Período: {df["Year"].min()} – {df["Year"].max()}')

Dataset carregado: 127,621 observações | 21,925 empresas
Colunas disponíveis: ['NIF Code', 'Company Name', 'Date of Establishment', 'Status', 'Status Date', 'BoardSize', 'IndependenceIndicator', 'OwnershipConcentration', 'CAE', 'District', 'LegalForm', 'Year', 'Revenue', 'Employees', 'TotalAssets', 'Equity', 'ROA', 'ROE', 'Liquidity', 'Indebtedness', 'NetProfit', 'EBITDA', 'Interests', 'RetainedEarnings', 'PMR', 'PMP', 'AcidTest', 'CashFlow', 'CAE_Sector', 'FirmAge']
Período: 2002 – 2025


## 1. Conversão e Limpeza de Dados SABI

Convertemos as colunas financeiras para formato numérico.

In [12]:
# Ajustado para os nomes reais detetados no ficheiro
fin_cols = ['TotalAssets', 'Equity', 'NetProfit', 'EBITDA', 'Liquidity', 'Indebtedness', 'Revenue', 'Interests']

for col in fin_cols:
    if col in df.columns:
        df[col] = safe_float(df[col])

print("Conversão numérica concluída.")

Conversão numérica concluída.


## 2. Refinamento de Rácios Financeiros

Complementamos os rácios existentes com métricas de rentabilidade e estrutura de capital.

In [13]:
# Liquidez (Já existe como 'Liquidity')
df['Current_Ratio'] = df['Liquidity']

# Solvabilidade / Leverage
df['Equity_to_Assets'] = df['Equity'] / df['TotalAssets'].replace(0, np.nan)
df['Debt_to_Assets'] = df['Indebtedness'] / 100.0  # Assumindo que Indebtedness está em %

# Rentabilidade
df['ROA_calc'] = df['NetProfit'] / df['TotalAssets'].replace(0, np.nan)
df['EBITDA_to_Assets'] = df['EBITDA'] / df['TotalAssets'].replace(0, np.nan)

# Eficiência
df['Asset_Turnover'] = df['Revenue'] / df['TotalAssets'].replace(0, np.nan)

print("Novos rácios calculados.")

Novos rácios calculados.


## 3. Tratamento de Outliers (Winsorization)

Removemos o ruído de valores extremos.

In [14]:
from scipy.stats.mstats import winsorize

ratio_cols = ['Current_Ratio', 'Equity_to_Assets', 'Debt_to_Assets', 'ROA_calc', 'EBITDA_to_Assets']

for col in ratio_cols:
    if col in df.columns:
        df[col] = winsorize(df[col].fillna(df[col].median()), limits=[0.01, 0.01])

print("Tratamento de outliers concluído.")

Tratamento de outliers concluído.


## 4. Exportação do Dataset de Features

In [15]:
df.to_csv(os.path.join(DATA_INTERIM, "micro_features.csv"), index=False)
print(f"Sucesso! Dataset de features guardado em: {os.path.join(DATA_INTERIM, 'micro_features.csv')}")

Sucesso! Dataset de features guardado em: ../data/interim\micro_features.csv
